# Delivery Performance — Customer Satisfaction

## Цель анализа

Цель этапа — оценить эффективность доставки и определить, как фактические сроки и соблюдение обещанной даты связаны с оценками клиентов.

В рамках анализа:
- рассчитываются фактическое время доставки и отклонение от ожидаемой даты;
- оценивается доля заказов, доставленных в срок;
- сравниваются оценки клиентов для разных групп доставки;
- статистически проверяется связь между сроками доставки и удовлетворённостью.

## 1. Подготовка данных

Для анализа используются таблицы:

- `orders` — статусы заказов, даты покупки, фактической и ожидаемой доставки;
- `reviews` — оценки клиентов.

В анализ включаются только доставленные заказы с корректными датами и оценкой. Если для одного заказа присутствует несколько записей об отзыве, используется последняя запись по времени ответа.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from scipy.stats import mannwhitneyu, spearmanr

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

In [ ]:
DATA_PATH = Path("../data")

orders = pd.read_csv(
    DATA_PATH / "olist_orders_dataset.csv"
)

reviews = pd.read_csv(
    DATA_PATH / "olist_order_reviews_dataset.csv"
)

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(
        orders[column],
        errors="coerce"
    )

reviews["review_answer_timestamp"] = pd.to_datetime(
    reviews["review_answer_timestamp"],
    errors="coerce"
)

duplicate_review_rows = reviews["order_id"].duplicated().sum()

reviews_one_per_order = (
    reviews
    .sort_values(
        ["order_id", "review_answer_timestamp"],
        na_position="first"
    )
    .drop_duplicates(
        subset="order_id",
        keep="last"
    )
)

delivered_orders = (
    orders
    .loc[orders["order_status"].eq("delivered")]
    .copy()
)

delivery = (
    delivered_orders
    .merge(
        reviews_one_per_order[
            ["order_id", "review_score"]
        ],
        on="order_id",
        how="inner",
        validate="one_to_one"
    )
)

rows_before_filter = len(delivery)

delivery = (
    delivery
    .dropna(
        subset=[
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "review_score"
        ]
    )
    .copy()
)

delivery["delivery_days"] = (
    delivery["order_delivered_customer_date"]
    - delivery["order_purchase_timestamp"]
).dt.total_seconds().div(86_400)

delivery["delay_days"] = (
    delivery["order_delivered_customer_date"]
    - delivery["order_estimated_delivery_date"]
).dt.total_seconds().div(86_400)

delivery = (
    delivery
    .loc[delivery["delivery_days"].ge(0)]
    .copy()
)

delivery["is_late"] = delivery["delay_days"].gt(0)

delivery["delivery_status"] = np.where(
    delivery["is_late"],
    "С опозданием",
    "В срок или раньше"
)

quality_summary = pd.DataFrame(
    {
        "Показатель": [
            "Доставленные заказы до фильтрации",
            "Заказы в итоговой выборке",
            "Исключённые записи",
            "Дополнительные строки отзывов по order_id"
        ],
        "Значение": [
            rows_before_filter,
            len(delivery),
            rows_before_filter - len(delivery),
            duplicate_review_rows
        ]
    }
)

quality_summary

Для показателя `delivery_days` используется интервал от покупки до фактического получения заказа.

Для оценки качества обещания клиенту дополнительно рассчитывается `delay_days`:

- значение ≤ 0 — заказ доставлен в срок или раньше;
- значение > 0 — заказ доставлен позже ожидаемой даты.

## 2. Основные показатели доставки

На данном этапе рассчитываются:
- среднее и медианное время доставки;
- 90-й перцентиль;
- доля заказов, доставленных в срок;
- доля заказов с опозданием.

In [ ]:
delivery_kpis = pd.DataFrame(
    {
        "Метрика": [
            "Количество заказов",
            "Среднее время доставки, дней",
            "Медианное время доставки, дней",
            "90-й перцентиль, дней",
            "Доля доставок в срок",
            "Доля доставок с опозданием",
            "Средняя оценка"
        ],
        "Значение": [
            f"{len(delivery):,}".replace(",", " "),
            f"{delivery['delivery_days'].mean():.1f}",
            f"{delivery['delivery_days'].median():.1f}",
            f"{delivery['delivery_days'].quantile(0.90):.1f}",
            f"{(~delivery['is_late']).mean():.1%}",
            f"{delivery['is_late'].mean():.1%}",
            f"{delivery['review_score'].mean():.2f}"
        ]
    }
)

delivery_kpis

In [ ]:
plot_limit = delivery["delivery_days"].quantile(0.99)

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(
    delivery.loc[
        delivery["delivery_days"].le(plot_limit),
        "delivery_days"
    ],
    bins=35
)

ax.set_title(
    "Распределение времени доставки"
)
ax.set_xlabel(
    "Время доставки, дней"
)
ax.set_ylabel(
    "Количество заказов"
)

plt.tight_layout()
plt.show()

На графике показаны значения до 99-го перцентиля, чтобы редкие экстремальные сроки не скрывали основную часть распределения. При расчёте метрик выбросы не исключаются.

In [ ]:
status_summary = (
    delivery
    .groupby(
        "delivery_status",
        observed=True
    )
    .agg(
        orders=("order_id", "nunique"),
        avg_review=("review_score", "mean"),
        median_review=("review_score", "median"),
        avg_delivery_days=("delivery_days", "mean")
    )
    .reindex(
        ["В срок или раньше", "С опозданием"]
    )
)

status_summary["orders_share"] = (
    status_summary["orders"]
    / status_summary["orders"].sum()
)

status_summary

## 3. Связь длительности доставки и оценки клиента

Для интерпретации заказы объединяются в группы по фактическому времени доставки.

Границы групп покрывают весь диапазон значений, включая доставку в день покупки и сроки свыше 30 дней.

In [ ]:
delivery["delivery_group"] = pd.cut(
    delivery["delivery_days"],
    bins=[
        -0.001,
        3,
        7,
        14,
        30,
        np.inf
    ],
    labels=[
        "0–3 дня",
        "4–7 дней",
        "8–14 дней",
        "15–30 дней",
        "Более 30 дней"
    ],
    include_lowest=True
)

delivery_group_summary = (
    delivery
    .groupby(
        "delivery_group",
        observed=True
    )
    .agg(
        orders=("order_id", "nunique"),
        avg_review=("review_score", "mean"),
        median_review=("review_score", "median"),
        low_rating_share=(
            "review_score",
            lambda values: values.le(2).mean()
        ),
        late_share=("is_late", "mean")
    )
)

delivery_group_summary

In [ ]:
plot_data = delivery_group_summary.reset_index()

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(
    plot_data["delivery_group"].astype(str),
    plot_data["avg_review"]
)

ax.set_title(
    "Средняя оценка по длительности доставки"
)
ax.set_xlabel(
    "Группа доставки"
)
ax.set_ylabel(
    "Средняя оценка"
)
ax.set_ylim(0, 5)

for bar, value in zip(
    bars,
    plot_data["avg_review"]
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.06,
        f"{value:.2f}",
        ha="center"
    )

plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 4. Соблюдение ожидаемой даты доставки

Абсолютная длительность зависит от расстояния и типа логистики. Поэтому отдельно оценивается отклонение от даты, обещанной клиенту.

Это позволяет отличить просто долгую, но ожидаемую доставку от фактического нарушения срока.

In [ ]:
delivery["delay_group"] = pd.cut(
    delivery["delay_days"],
    bins=[
        -np.inf,
        0,
        3,
        7,
        14,
        np.inf
    ],
    labels=[
        "В срок или раньше",
        "Опоздание 1–3 дня",
        "Опоздание 4–7 дней",
        "Опоздание 8–14 дней",
        "Опоздание более 14 дней"
    ]
)

delay_group_summary = (
    delivery
    .groupby(
        "delay_group",
        observed=True
    )
    .agg(
        orders=("order_id", "nunique"),
        avg_review=("review_score", "mean"),
        median_review=("review_score", "median"),
        low_rating_share=(
            "review_score",
            lambda values: values.le(2).mean()
        )
    )
)

delay_group_summary

In [ ]:
plot_data = delay_group_summary.reset_index()

fig, ax = plt.subplots(figsize=(11, 5))

bars = ax.bar(
    plot_data["delay_group"].astype(str),
    plot_data["avg_review"]
)

ax.set_title(
    "Средняя оценка по отклонению от ожидаемой даты"
)
ax.set_xlabel(
    "Соблюдение ожидаемого срока"
)
ax.set_ylabel(
    "Средняя оценка"
)
ax.set_ylim(0, 5)

for bar, value in zip(
    bars,
    plot_data["avg_review"]
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.06,
        f"{value:.2f}",
        ha="center"
    )

plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 5. Статистическая проверка

Оценка клиента задана порядковой шкалой от 1 до 5, поэтому для сравнения заказов, доставленных в срок и с опозданием, используется непараметрический тест Манна–Уитни.

Дополнительно рассчитывается корреляция Спирмена:
- между фактическим временем доставки и оценкой;
- между отклонением от ожидаемой даты и оценкой.

Уровень значимости: α = 0.05.

In [ ]:
on_time_scores = delivery.loc[
    ~delivery["is_late"],
    "review_score"
]

late_scores = delivery.loc[
    delivery["is_late"],
    "review_score"
]

u_statistic, p_value = mannwhitneyu(
    late_scores,
    on_time_scores,
    alternative="two-sided"
)

rank_biserial = (
    2 * u_statistic
    / (len(late_scores) * len(on_time_scores))
    - 1
)

delivery_spearman, delivery_spearman_p = spearmanr(
    delivery["delivery_days"],
    delivery["review_score"]
)

delay_spearman, delay_spearman_p = spearmanr(
    delivery["delay_days"],
    delivery["review_score"]
)

statistical_results = pd.DataFrame(
    {
        "Показатель": [
            "U-статистика",
            "p-value теста Манна–Уитни",
            "Рангово-бисериальная корреляция",
            "Спирмен: время доставки и оценка",
            "p-value корреляции времени доставки",
            "Спирмен: опоздание и оценка",
            "p-value корреляции опоздания"
        ],
        "Значение": [
            u_statistic,
            p_value,
            rank_biserial,
            delivery_spearman,
            delivery_spearman_p,
            delay_spearman,
            delay_spearman_p
        ]
    }
)

statistical_results

## 6. Business Insights

In [ ]:
fast_rating = delivery_group_summary.loc[
    "0–3 дня",
    "avg_review"
]

long_rating = delivery_group_summary.loc[
    "Более 30 дней",
    "avg_review"
]

on_time_rating = status_summary.loc[
    "В срок или раньше",
    "avg_review"
]

late_rating = status_summary.loc[
    "С опозданием",
    "avg_review"
]

significance_text = (
    "статистически значимы"
    if p_value < 0.05
    else "не являются статистически значимыми"
)

insights = f"""
Основные выводы:

1. Среднее время доставки составляет **{delivery['delivery_days'].mean():.1f} дня**, медианное — **{delivery['delivery_days'].median():.1f} дня**.

2. В срок или раньше доставлено **{(~delivery['is_late']).mean():.1%}** заказов. Доля опоздавших заказов составляет **{delivery['is_late'].mean():.1%}**.

3. Средняя оценка снижается с **{fast_rating:.2f}** для доставки за 0–3 дня до **{long_rating:.2f}** для доставки более 30 дней.

4. Заказы, доставленные в срок, получают в среднем **{on_time_rating:.2f}**, а заказы с опозданием — **{late_rating:.2f}**.

5. Различия между оценками заказов, доставленных в срок и с опозданием, {significance_text} (`p-value = {p_value:.3g}`, рангово-бисериальная корреляция = {rank_biserial:.3f}).

6. Корреляция Спирмена между величиной опоздания и оценкой составляет **{delay_spearman:.3f}**. Отрицательное значение означает, что при росте опоздания оценки в среднем снижаются.

### Рекомендации

- использовать долю доставок в срок как основной KPI логистики;
- отдельно контролировать заказы с опозданием более 7 и 14 дней;
- выявить регионы и продавцов, формирующих наибольшую долю задержек;
- не интерпретировать обнаруженную связь как доказанную причинность без контроля категории товара, региона, цены и других факторов.
"""

display(Markdown(insights))